In [ ]:
"""
CV grid for Persistence Images (H0 y H1)

Output:
  - grid_PI
"""

import os
import numpy as np
import pandas as pd

from gudhi.representations import PersistenceImage
from gudhi.representations.preprocessing import BirthPersistenceTransform

from scipy.stats import mannwhitneyu, combine_pvalues
from scipy.ndimage import gaussian_filter
from sklearn.model_selection import RepeatedStratifiedKFold

# Configuration
BASE = "RIPS"
DIM_INTERVALS = [0, 1]

PI_RESOLUTIONS = [(10, 10), (25, 25), (50, 50)]
BANDWIDTHS = [0.5, 2.0]
WEIGHTS = ["const", "pers"]

IMG_SIGMAS = [0, 2]
NORMALIZATIONS = ["none", "l1"]
THR_OPTIONS = ["0", "p10"]

EPS = 1e-12

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 0

OUTDIR = "PI_MWU"
os.makedirs(OUTDIR, exist_ok=True)

BP = BirthPersistenceTransform()

# LOADER
def read_and_save(filedir, tube):
    if tube and tube[0] != '.':
        _, ext = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split('_')[-1].split('.')[0]

        if ext != '.pdf' and tubenamerips == 'Rips0':
            r0_path = os.path.join(filedir, '_'.join(tube.split('_')[:-1]) + '_Rips0.txt')
            r1_path = os.path.join(filedir, '_'.join(tube.split('_')[:-1]) + '_Rips1.txt')

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []

def list_patients(root_dir):
    return [x for x in sorted(os.listdir(root_dir)) if not x.startswith(".")]

def find_rips0_file(patient_dir):
    for f in sorted(os.listdir(patient_dir)):
        if f.startswith("."):
            continue
        if f.endswith("_Rips0.txt"):
            return f
    return None

def load_group_diagrams(group_root):
    out = {}
    for patient in list_patients(group_root):
        p_dir = os.path.join(group_root, patient)
        rips0_file = find_rips0_file(p_dir)
        if rips0_file is None:
            continue
        data = read_and_save(p_dir, rips0_file)
        if not data:
            continue
        diagrams = data[0]
        if diagrams is None or len(diagrams) < 2:
            continue
        out[patient] = diagrams
    return out

def collect_persistences(diagrams_list, dim):
    pers = []
    for diagrams in diagrams_list:
        pairs = diagrams[dim]
        pairs = np.asarray(pairs, dtype=float)
        p = pairs[:, 1] - pairs[:, 0]
        p = p[np.isfinite(p)]
        p = p[p > 0]
        if p.size:
            pers.append(p)
    return np.concatenate(pers) if pers else np.array([])

def apply_persistence_threshold(pairs, thr):
    pairs = np.asarray(pairs, float)
    pers = pairs[:, 1] - pairs[:, 0]
    keep = np.isfinite(pers) & (pers >= thr)
    return pairs[keep]

def normalize_vec(v, mode):
    if mode == "l1":
        return v / (np.sum(np.abs(v)) + EPS)
    return v

def smooth_image(v, res, sigma):
    if sigma <= 0:
        return v
    img = v.reshape(res)
    img = gaussian_filter(img, sigma=sigma)
    return img.ravel()

def mannwhitney_per_bin(X0, X1):
    pvals = []
    for j in range(X0.shape[1]):
        _, p = mannwhitneyu(X0[:, j], X1[:, j])
        pvals.append(p)
    return np.array(pvals)

def fisher_combine(pvals):
    pvals = np.clip(pvals, EPS, 1)
    return combine_pvalues(pvals)[1]

# Main
if __name__ == "__main__":

    NR = load_group_diagrams(os.path.join(BASE, "NonRelapse"))
    R  = load_group_diagrams(os.path.join(BASE, "Relapse"))

    X = list(NR.values()) + list(R.values())
    y = np.array([0]*len(NR) + [1]*len(R))

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE
    )

    rows = []

    for dim in DIM_INTERVALS:
        for thr_opt in THR_OPTIONS:
            for weight in WEIGHTS:
                for sigma in IMG_SIGMAS:
                    for res in PI_RESOLUTIONS:
                        for bw in BANDWIDTHS:
                            for norm in NORMALIZATIONS:

                                scores = []

                                for tr, te in cv.split(np.zeros(len(y)), y):

                                    X_tr = [X[i] for i in tr]
                                    X_te = [X[i] for i in te]
                                    y_te = y[te]

                                    if thr_opt == "0":
                                        thr = 0
                                    else:
                                        pers = collect_persistences(X_tr, dim)
                                        thr = np.percentile(pers,10) if len(pers) else 0

                                    pi = PersistenceImage(
                                        bandwidth=bw,
                                        resolution=list(res)
                                    )

                                    X0, X1 = [], []

                                    for d, lab in zip(X_te, y_te):
                                        v = pi(BP(apply_persistence_threshold(d[dim], thr)))
                                        v = normalize_vec(v, norm)
                                        v = smooth_image(v, res, sigma)

                                        if lab == 0:
                                            X0.append(v)
                                        else:
                                            X1.append(v)

                                    if len(X0) < 2 or len(X1) < 2:
                                        score = 0
                                    else:
                                        X0 = np.vstack(X0)
                                        X1 = np.vstack(X1)
                                        pvals = mannwhitney_per_bin(X0, X1)
                                        score = -np.log10(max(fisher_combine(pvals), EPS))

                                    scores.append(score)

                                rows.append({
                                    "dimension": dim,
                                    "thr_opt": thr_opt,
                                    "weight": weight,
                                    "sigma": sigma,
                                    "resolution": str(res),
                                    "bandwidth": bw,
                                    "normalization": norm,
                                    "score_median_CV": np.median(scores),
                                    "score_mean_CV": np.mean(scores)
                                })

    df = pd.DataFrame(rows).sort_values(
        ["dimension", "score_median_CV"],
        ascending=[True, False]
    )

    out = os.path.join(OUTDIR, "grid_PI.csv")
    df.to_csv(out, index=False)

    print("Saved:", out)